In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [3]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import sys
import os

# The user specified the path: My Drive/EPA/MLaBP
# Construct the full absolute path within the mounted Drive
notebook_dir_in_drive = '/content/drive/MyDrive/EPA/MLaBP'

# Add this directory to the beginning of sys.path if it's not already there
if notebook_dir_in_drive not in sys.path:
    sys.path.insert(0, notebook_dir_in_drive)
    print(f"Added user-specified directory to sys.path: {notebook_dir_in_drive}")
else:
    print(f"User-specified directory already in sys.path: {notebook_dir_in_drive}")

from tokenizer import CharachterLevelTokenizer, TiktokenTokenizer, MinbpeTokenizer

In [ ]:
MINBPE_MAX_CHARS = 100000
DATASETS = ['shakespeare', 'text8']
TOKENIZER_NAMES = ["CharacterLevel", "Tiktoken (cl100k)", "MinBPE"]
cache_keys = ['char', 'tiktoken', 'minbpe']
cache_dir = os.path.join(notebook_dir_in_drive, 'data', 'cache')

# Tiktoken exceeds GPU memory on text8 — skip it for that dataset
SKIP = {
    'text8': {'Tiktoken (cl100k)'},
}

In [ ]:
base_params = {
    'batch_size' : 64,
    'block_size' : 256,
    'max_iters' : 10000,
    'eval_interval' : 500,
    'learning_rate' : 3e-4,
    'eval_iters' : 200,
    'n_embd' : 384,
    'n_head' : 6,
    'n_layer': 6,
    'dropout' : 0.2
}

device = 'cuda' if torch.cuda.is_available() else 'cpu'

torch.manual_seed(42)

In [9]:
print(device)

cuda


In [10]:
def get_batch(data, params, device):
    ix = torch.randint(len(data) - params['block_size'], (params['batch_size'],))
    x = torch.stack([data[i:i+params['block_size']] for i in ix])
    y = torch.stack([data[i+1:i+params['block_size']+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [11]:
@torch.no_grad()
def estimate_loss(model, data, params, device):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(params['eval_iters'])
        for k in range(params['eval_iters']):
            X, Y = get_batch(data[split], params, device)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [12]:
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size, params):
        super().__init__()
        self.key = nn.Linear(params['n_embd'], head_size, bias=False)
        self.query = nn.Linear(params['n_embd'], head_size, bias=False)
        self.value = nn.Linear(params['n_embd'], head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(params['block_size'], params['block_size'])))

        self.dropout = nn.Dropout(params['dropout'])

    def forward(self, x):
        # input of size (batch, time-step, channels)
        # output of size (batch, time-step, head size)
        B,T,C = x.shape
        k = self.key(x)   # (B,T,hs)
        q = self.query(x) # (B,T,hs)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 # (B, T, hs) @ (B, hs, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,hs)
        out = wei @ v # (B, T, T) @ (B, T, hs) -> (B, T, hs)
        return out

In [13]:
class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, params, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size, params) for _ in range(params['n_head'])])
        self.proj = nn.Linear(head_size * params['n_head'], params['n_embd'])
        self.dropout = nn.Dropout(params['dropout'])

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

In [14]:
class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, params):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(params['n_embd'], 4 * params['n_embd']),
            nn.ReLU(),
            nn.Linear(4 * params['n_embd'], params['n_embd']),
            nn.Dropout(params['dropout']),
        )

    def forward(self, x):
        return self.net(x)

In [15]:
class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, params):
        super().__init__()
        head_size = params['n_embd'] // params['n_head']
        self.sa = MultiHeadAttention(params, head_size)
        self.ffwd = FeedFoward(params)
        self.ln1 = nn.LayerNorm(params['n_embd'])
        self.ln2 = nn.LayerNorm(params['n_embd'])

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [16]:
class GPTLanguageModel(nn.Module):

    def __init__(self, params):
        super().__init__()
        self.params = params
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(params['vocab_size'], params['n_embd'])
        self.position_embedding_table = nn.Embedding(params['block_size'], params['n_embd'])
        self.blocks = nn.Sequential(*[Block(params) for _ in range(params['n_layer'])])
        self.ln_f = nn.LayerNorm(params['n_embd']) # final layer norm
        self.lm_head = nn.Linear(params['n_embd'], params['vocab_size'])

        # better init, not covered in the original GPT video, but important, will cover in followup video
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -self.params['block_size']:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [ ]:
import time
from tqdm import tqdm

def run_experiment(tokenizer, name, text, cache_name):
    # --- Encoding (cached) ---
    cache_file = os.path.join(cache_dir, f'{cache_name}.pt')

    if os.path.exists(cache_file):
        print(f"[cache] Loading encoded tokens from {cache_file}")
        payload = torch.load(cache_file)
        encoded = payload['encoded']
        vocab_size = payload['vocab_size']
    else:
        print(f"[cache] No cache found, encoding with {name} tokenizer...")
        encoded = torch.tensor(tokenizer.encode(text), dtype=torch.long)
        vocab_size = len(tokenizer.vocab)
        torch.save({'encoded': encoded, 'vocab_size': vocab_size}, cache_file)
        print(f"[cache] Saved to {cache_file}")

    params = {**base_params, 'vocab_size': vocab_size}

    # Train/val split
    n = int(0.9 * len(encoded))
    data = {'train': encoded[:n], 'val': encoded[n:]}

    # Model + optimizer
    torch.manual_seed(42)
    model = GPTLanguageModel(params).to(device)
    print(f"\n=== {name} | vocab_size={vocab_size} | tokens={len(encoded)} | {sum(p.numel() for p in model.parameters())/1e6:.2f}M parameters ===")
    optimizer = torch.optim.AdamW(model.parameters(), lr=params['learning_rate'])

    history = []
    t0 = time.time()
    pbar = tqdm(range(params['max_iters'] + 1), desc=name)
    for iter in pbar:
        if iter % params['eval_interval'] == 0:
            losses = estimate_loss(model, data, params, device)
            pbar.set_postfix(train=f"{losses['train']:.4f}", val=f"{losses['val']:.4f}")
            tqdm.write(f"  step {iter:4d}: train {losses['train']:.4f}  val {losses['val']:.4f}")
            history.append({'step': iter, 'train': losses['train'].item(), 'val': losses['val'].item()})
        if iter < params['max_iters']:
            xb, yb = get_batch(data['train'], params, device)
            _, loss = model(xb, yb)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
    t1 = time.time()

    return model, history, t1 - t0

In [ ]:
import math

def compute_metrics(history, vocab_size):
    log_v = math.log(vocab_size)
    enriched = []
    for h in history:
        enriched.append({
            'step': h['step'],
            'train_loss': h['train'],
            'val_loss': h['val'],
            'train_ppl': math.exp(h['train']),
            'val_ppl': math.exp(h['val']),
            'train_norm_loss': h['train'] / log_v,
            'val_norm_loss': h['val'] / log_v,
            'train_norm_ppl': math.exp(h['train'] / log_v),
            'val_norm_ppl': math.exp(h['val'] / log_v),
        })
    return enriched

In [ ]:
import json
import matplotlib.pyplot as plt
from datetime import datetime
from huggingface_hub import hf_hub_download

def plot_metric(all_metrics, active_names, metric_train, metric_val, title, ylabel, save_path):
    fig, ax = plt.subplots(figsize=(10, 5))
    for metrics, name in zip(all_metrics, active_names):
        steps = [h['step'] for h in metrics]
        ax.plot(steps, [h[metric_train] for h in metrics], label=f'{name} train')
        ax.plot(steps, [h[metric_val]   for h in metrics], label=f'{name} val', linestyle='--')
    ax.set_title(title); ax.set_xlabel('step'); ax.set_ylabel(ylabel); ax.legend()
    plt.tight_layout()
    plt.savefig(save_path)
    plt.show()

all_results = {}

for DATASET in DATASETS:
    print(f"\n{'='*60}\n  DATASET: {DATASET.upper()}\n{'='*60}")

    # --- Load text ---
    if DATASET == "shakespeare":
        with open('/content/drive/MyDrive/EPA/MLaBP/data/input.txt', 'r', encoding='utf-8') as f:
            text = f.read()
    elif DATASET == "text8":
        sentences_path = hf_hub_download(repo_id="roshbeed/text8-dataset", filename="text8_sentences.json")
        with open(sentences_path, 'r') as f:
            _data = json.load(f)
        text = ' '.join(_data['sentences'])

    print(f"Length: {len(text):,} chars | preview: {text[:80]!r}")

    # --- Active tokenizers (respect SKIP) ---
    skip_set = SKIP.get(DATASET, set())
    active_names = [n for n in TOKENIZER_NAMES if n not in skip_set]
    active_keys  = [k for n, k in zip(TOKENIZER_NAMES, cache_keys) if n not in skip_set]

    tokenizers = []
    for name in active_names:
        if name == "CharacterLevel":
            tokenizers.append(CharachterLevelTokenizer(text))
        elif name == "Tiktoken (cl100k)":
            tokenizers.append(TiktokenTokenizer(text))
        elif name == "MinBPE":
            tokenizers.append(MinbpeTokenizer(text, max_chars=MINBPE_MAX_CHARS))

    for tok, name in zip(tokenizers, active_names):
        print(f"{name} vocab: {len(tok.vocab)}")

    # --- Run experiments ---
    results = [
        run_experiment(tok, name, text, f"{key}_{DATASET}")
        for tok, name, key in zip(tokenizers, active_names, active_keys)
    ]
    models    = [r[0] for r in results]
    histories = [r[1] for r in results]
    times     = [r[2] for r in results]

    # --- Compute derived metrics ---
    all_metrics = [compute_metrics(h, len(tok.vocab)) for h, tok in zip(histories, tokenizers)]

    experiments = list(zip(active_names, all_metrics, [len(t.vocab) for t in tokenizers], times))

    # --- Summary table ---
    header = f"{'Tokenizer':<20} {'Vocab':>6}  {'TrainLoss':>9} {'ValLoss':>8}  {'TrainPPL':>9} {'ValPPL':>8}  {'NormValLoss':>12} {'NormValPPL':>11}  {'Time(s)':>8}"
    sep = "-" * len(header)
    print(f"\n{sep}\n{header}\n{sep}")

    summary_rows = []
    for name, metrics, vocab_size, train_time in experiments:
        last = metrics[-1]
        print(
            f"{name:<20} {vocab_size:>6}  "
            f"{last['train_loss']:>9.4f} {last['val_loss']:>8.4f}  "
            f"{last['train_ppl']:>9.2f} {last['val_ppl']:>8.2f}  "
            f"{last['val_norm_loss']:>12.4f} {last['val_norm_ppl']:>11.4f}  "
            f"{train_time:>8.1f}"
        )
        summary_rows.append({
            'tokenizer': name, 'vocab_size': vocab_size,
            'train_loss': last['train_loss'], 'val_loss': last['val_loss'],
            'train_ppl': last['train_ppl'], 'val_ppl': last['val_ppl'],
            'train_norm_loss': last['train_norm_loss'], 'val_norm_loss': last['val_norm_loss'],
            'train_norm_ppl': last['train_norm_ppl'], 'val_norm_ppl': last['val_norm_ppl'],
            'training_time_s': train_time,
        })
    print(sep)

    # --- Run folder ---
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    save_dir = os.path.join(notebook_dir_in_drive, f'models/attention/{DATASET}_{timestamp}')
    os.makedirs(save_dir, exist_ok=True)
    print(f"\nRun folder: {save_dir}")

    # --- Plots ---
    fig, axes = plt.subplots(1, len(active_names), figsize=(6 * len(active_names), 4), sharey=False)
    if len(active_names) == 1:
        axes = [axes]
    for ax, history, name in zip(axes, histories, active_names):
        steps = [h['step'] for h in history]
        ax.plot(steps, [h['train'] for h in history], label='train')
        ax.plot(steps, [h['val']   for h in history], label='val', linestyle='--')
        ax.set_title(name); ax.set_xlabel('step'); ax.set_ylabel('loss'); ax.legend()
    plt.suptitle(f'GPT Model [{DATASET}]: Loss by Tokenizer')
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, 'plot_loss_curves.png'))
    plt.show()

    plot_metric(all_metrics, active_names, 'train_ppl', 'val_ppl',
                f'GPT Model [{DATASET}]: Perplexity by Tokenizer', 'PPL',
                os.path.join(save_dir, 'plot_ppl_curves.png'))
    plot_metric(all_metrics, active_names, 'train_norm_loss', 'val_norm_loss',
                f'GPT Model [{DATASET}]: Normalized Loss (loss / log|V|) by Tokenizer', 'Normalized Loss',
                os.path.join(save_dir, 'plot_normalized_loss.png'))
    plot_metric(all_metrics, active_names, 'train_norm_ppl', 'val_norm_ppl',
                f'GPT Model [{DATASET}]: Normalized PPL by Tokenizer', 'Normalized PPL',
                os.path.join(save_dir, 'plot_normalized_ppl.png'))

    # --- Text generation samples ---
    context = torch.zeros((1, 1), dtype=torch.long, device=device)
    for model, tok, name in zip(models, tokenizers, active_names):
        print(f"\n=== [{DATASET}] {name} generation ===")
        print(tok.decode(model.generate(context, max_new_tokens=500)[0].tolist()))

    # --- Save ---
    for model, key in zip(models, active_keys):
        torch.save(model.state_dict(), os.path.join(save_dir, f'{key}_model.pt'))
    with open(os.path.join(save_dir, 'base_params.json'), 'w') as f:
        json.dump(base_params, f, indent=4)
    with open(os.path.join(save_dir, 'summary.json'), 'w') as f:
        json.dump(summary_rows, f, indent=4)
    print(f"All outputs saved to {save_dir}/")

    all_results[DATASET] = {'summary': summary_rows, 'metrics': all_metrics}

print(f"\n{'='*60}\n  ALL DATASETS COMPLETE\n{'='*60}")